# 13 — Tabular Export

**Purpose:** Show every way to get TorchLens data into tabular form: `Trace.to_pandas()`,
per-record `to_pandas()` (Op/Layer), and the `torchlens.export` submodule
(`csv`, `json`, `parquet`).  Show schema, column counts, and a read-back round-trip.

**Surfaces covered:**
- [ ] `torchlens.export` submodule — confirmed exists at `torchlens.export`
- [ ] `torchlens.export.csv(trace, path)` — write CSV, read back with pandas
- [ ] `torchlens.export.json(trace, path)` — write JSON (records orient), read back
- [ ] `torchlens.export.parquet(trace, path)` — write Parquet, read back
- [ ] `Trace.to_pandas()` — full trace as multi-row DataFrame (one row = one Op)
- [ ] `Layer.to_pandas()` — layer-aggregated single-row DataFrame
- [ ] `Op.to_pandas()` — pass-level single-row DataFrame (more columns than Layer)
- [ ] Schema comparison: column counts across Trace/Layer/Op DataFrames
- [ ] Key columns: `layer_label`, `layer_type`, `activation_memory`
- [ ] Round-trip: write -> read back -> confirm shape and key values

## 1. Environment setup

In [ ]:
import pathlib
import sys
import os
import tempfile
import json as _json

sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torchlens as tl
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

TMPDIR = tempfile.mkdtemp(prefix="tl_audit_13_")
print(f"temp dir          : {TMPDIR}")

## 2. Verify `torchlens.export` module path

In [ ]:
# Verify the module exists and locate it
try:
    import torchlens.export as tl_export

    print("torchlens.export EXISTS")
    print("module file:", tl_export.__file__)
    print()
    # Show public names
    public = (
        tl_export.__all__
        if hasattr(tl_export, "__all__")
        else [n for n in dir(tl_export) if not n.startswith("_")]
    )
    print("Public names:", sorted(public))
except ImportError as e:
    print(f"\u26a0\ufe0f  GAP: torchlens.export NOT importable: {e}")

In [ ]:
import inspect

# Confirm csv/json/parquet are callable with correct signatures
for name in ["csv", "json", "parquet"]:
    fn = getattr(tl_export, name, None)
    if fn is None:
        print(f"\u26a0\ufe0f  GAP: torchlens.export.{name} not found")
    else:
        sig = inspect.signature(fn)
        print(f"torchlens.export.{name}{sig}")

## 3. Capture a trace for all export demos

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

print(repr(trace))
print("layer_labels:", trace.layer_labels)

## 4. `Trace.to_pandas()` — full trace DataFrame

In [ ]:
trace_df = trace.to_pandas()

print("trace.to_pandas() shape:", trace_df.shape)
print(f"  {trace_df.shape[0]} rows (one per Op)  x  {trace_df.shape[1]} columns")
print()
print("All columns:")
# Print in groups of 10 for readability
cols = list(trace_df.columns)
for i in range(0, len(cols), 10):
    print(" ", cols[i : i + 10])

In [ ]:
# Show key columns
key_cols = [
    c
    for c in ["layer_label", "layer_type", "activation_memory", "label", "func", "grad_fn_name"]
    if c in trace_df.columns
]
print("Key column view:")
print(trace_df[key_cols].to_string())

## 5. `Layer.to_pandas()` — layer-aggregated view

In [ ]:
layer = trace.layers["relu_1_2"]
layer_df = layer.to_pandas()

print("Layer.to_pandas() shape:", layer_df.shape)
print(f"  1 row (this layer)  x  {layer_df.shape[1]} columns")
print()
# Show first 12 columns
print("First 12 columns:")
print(layer_df.iloc[:, :12].to_string())

## 6. `Op.to_pandas()` — pass-level view

In [ ]:
op = trace.ops["relu_1_2:1"]
op_df = op.to_pandas()

print("Op.to_pandas() shape:", op_df.shape)
print(f"  1 row (this op/pass)  x  {op_df.shape[1]} columns")
print()
# Show first 12 columns
print("First 12 columns:")
print(op_df.iloc[:, :12].to_string())

## 7. Schema comparison — column counts across DataFrame levels

In [ ]:
print("=== DataFrame schema comparison ===")
print(f"  Trace.to_pandas()    : {len(trace_df.columns):4d} columns  ({trace_df.shape[0]} rows)")
print(f"  Layer.to_pandas()    : {len(layer_df.columns):4d} columns  (1 row per layer)")
print(f"  Op.to_pandas()       : {len(op_df.columns):4d} columns  (1 row per pass)")
print()
print("Note: Trace has a DIFFERENT column set from both Layer and Op.")
print("  Columns in Trace but not Op :", len(set(trace_df.columns) - set(op_df.columns)))
print("  Columns in Op but not Trace  :", len(set(op_df.columns) - set(trace_df.columns)))
print("  Shared columns               :", len(set(trace_df.columns) & set(op_df.columns)))

## 8. `torchlens.export.csv` — write CSV and read back

In [ ]:
csv_path = os.path.join(TMPDIR, "trace.csv")
written_path = tl_export.csv(trace, csv_path)

print(f"csv() wrote to: {written_path}")
print(f"file size: {os.path.getsize(csv_path):,} B")

In [ ]:
# Read back with pandas
import pandas as pd

csv_df = pd.read_csv(csv_path)
print("Round-trip CSV shape:", csv_df.shape)
print()

# Verify key column values survived
key_cols = [c for c in ["layer_label", "layer_type", "activation_memory"] if c in csv_df.columns]
print("Key column round-trip:")
print(csv_df[key_cols].to_string())
print()
print("layer_label matches original:", list(csv_df["layer_label"]) == trace.layer_labels)

## 9. `torchlens.export.json` — write JSON and read back

In [ ]:
json_path = os.path.join(TMPDIR, "trace.json")
tl_export.json(trace, json_path)

print(f"json() wrote to: {json_path}")
print(f"file size: {os.path.getsize(json_path):,} B")

with open(json_path) as f:
    json_data = _json.load(f)

print(f"JSON type: {type(json_data).__name__}")
print(f"Records: {len(json_data)}  (one per Op)")
print()
# Show first record's keys
if json_data:
    first_keys = list(json_data[0].keys())
    print(f"First record has {len(first_keys)} fields:")
    print(" ", first_keys[:10], "...")

In [ ]:
# Read back with pandas via orient='records'
json_df = pd.read_json(json_path, orient="records")
print("Round-trip JSON shape:", json_df.shape)
key_cols = [c for c in ["layer_label", "layer_type", "activation_memory"] if c in json_df.columns]
print(json_df[key_cols].to_string())

## 10. `torchlens.export.parquet` — write Parquet and read back

In [ ]:
parquet_path = os.path.join(TMPDIR, "trace.parquet")

try:
    tl_export.parquet(trace, parquet_path)
    print(f"parquet() wrote to: {parquet_path}")
    print(f"file size: {os.path.getsize(parquet_path):,} B")
except ImportError as e:
    print(f"\u26a0\ufe0f  parquet requires pyarrow/fastparquet: {e}")
    parquet_path = None

In [ ]:
if parquet_path and os.path.exists(parquet_path):
    parquet_df = pd.read_parquet(parquet_path)
    print("Round-trip Parquet shape:", parquet_df.shape)
    key_cols = [
        c for c in ["layer_label", "layer_type", "activation_memory"] if c in parquet_df.columns
    ]
    print(parquet_df[key_cols].to_string())
    print()
    print("layer_label matches original:", list(parquet_df["layer_label"]) == trace.layer_labels)

## 11. Demo on a more complex model (demo_model)

In [ ]:
demo_model, demo_x = ZOO["demo_model"]()
demo_trace = tl.trace(demo_model, demo_x)

print(repr(demo_trace))
demo_df = demo_trace.to_pandas()
print(f"demo trace.to_pandas() shape: {demo_df.shape}")
print()

key_cols = [c for c in ["layer_label", "layer_type", "activation_memory"] if c in demo_df.columns]
print(demo_df[key_cols].to_string())

## 12. Additional export formats available in `torchlens.export`

The submodule has many more formats beyond the core three.

In [ ]:
# List all export functions with their signatures
all_public = sorted(n for n in dir(tl_export) if not n.startswith("_"))
print("All public names in torchlens.export:")
for name in all_public:
    obj = getattr(tl_export, name)
    if callable(obj):
        try:
            sig = str(inspect.signature(obj))
            # Truncate long signatures
            if len(sig) > 60:
                sig = sig[:57] + "..."
            print(f"  {name}{sig}")
        except (ValueError, TypeError):
            print(f"  {name}  (not inspectable)")

---

## ⚠️ GAPs / ergonomic smells

- **`torchlens.export` EXISTS** at `torchlens/export/__init__.py` and is importable as
  `import torchlens.export as tl_export`.  The module is NOT re-exported under `tl.*`
  (not in `tl.__all__`), so users must know to import it explicitly.

- **Three DataFrame schemas, none a superset of another.** `Trace.to_pandas()` (153 cols),
  `Layer.to_pandas()` (86 cols), `Op.to_pandas()` (181 cols) have overlapping but distinct
  column sets.  A user who joins on `layer_label` across these will hit silent column
  name conflicts.  No shared schema constant is exposed.

- **`forward_flops` and `out_shape` are NOT columns in `Trace.to_pandas()`** despite being
  natural things to want.  `activation_memory` IS present.  Users expecting `out_shape`
  from the table will need to compute from Op records individually.

- **`torchlens.export.json` defaults to `orient='records'`** which is good for
  per-row consumption, but the `orient` kwarg is exposed and forwarded to
  `DataFrame.to_json` — the docs should mention valid orient values.

- **Parquet requires a third-party engine** (`pyarrow` or `fastparquet`).  This is not
  explicitly surfaced in the function signature/docstring.  A plain `ImportError` is
  raised if neither is available; a more targeted message would help.

- **`torchlens.export` is a package directory** (`torchlens/export/__init__.py`), not
  a single-file module.  This is fine but means `tl_export.__file__` shows the `__init__.py`
  path — the real implementations are in submodules inside `export/`.